In [11]:
# !pip install requests

import requests
from xml.etree import ElementTree
import json
import pprint

In [14]:
#asked ChatGPT how to tweak the API call so it is useable in python

# Define parameters
base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "pubmed",
    "term": "Alzheimers AND 2024[pdat]",
    "retmax": "1000",
    "retmode": "xml"
}

# Send request
response = requests.get(base_url, params=params)
root = ElementTree.fromstring(response.text)

# Extract article IDs
ids = [id_elem.text for id_elem in root.findall(".//Id")]
print(f"Found {len(ids)} articles.")
print(ids[:10])  # show first 10 IDs

Found 1000 articles.
['41058862', '41024939', '40980211', '40973408', '40973407', '40973404', '40973402', '40973401', '40973397', '40970099']


In [15]:
# found all 1000 cancer papers

# Define parameters
base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "pubmed", # from PubMed
    "term": "cancer AND 2024[pdat]", #cancer
    "retmax": "1000", # only send 1000 articles
    "retmode": "xml" # sends in xml format
}

# Send request
response = requests.get(base_url, params=params)
root = ElementTree.fromstring(response.text)

# Extract article IDs
ids = [id_elem.text for id_elem in root.findall(".//Id")]
print(f"Found {len(ids)} articles.")
print(ids[:10])  # show first 10 IDs


Found 1000 articles.
['41104208', '41099068', '41098208', '41090149', '41090146', '41090144', '41089424', '41089265', '41084562', '41084543']


In [16]:
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
# Note: PubMed limits ~200 IDs per efetch call
all_metadata = []

for i in range(0, len(ids), 200):
    batch_ids = ",".join(ids[i:i+200])
    fetch_params = {
        "db": "pubmed",
        "id": batch_ids,
        "retmode": "xml"
    }
    fetch_response = requests.get(fetch_url, params=fetch_params)
    root = ElementTree.fromstring(fetch_response.text)

    for article in root.findall(".//PubmedArticle"):
        title_elem = article.find(".//ArticleTitle")
        abstract_elem = article.find(".//Abstract/AbstractText")
        journal_elem = article.find(".//Journal/Title")
        date_elem = article.find(".//PubDate/Year")

        metadata = {
            "title": title_elem.text if title_elem is not None else None,
            "abstract": abstract_elem.text if abstract_elem is not None else None,
            "journal": journal_elem.text if journal_elem is not None else None,
            "year": date_elem.text if date_elem is not None else None
        }
        all_metadata.append(metadata)

print(f"Retrieved metadata for {len(all_metadata)} articles.")
print(all_metadata[:3]) 

Retrieved metadata for 999 articles.
[{'title': 'Neoadjuvant systemic therapy for hepatocellular carcinoma: challenges and opportunities-a narrative review.', 'abstract': 'Hepatocellular carcinoma (HCC) is one of the most common cancers with high mortality rate worldwide. Surgical resection, liver transplantation (LT), and thermal ablation are primary curative methods for early-stage HCC. However, the high recurrence rate following surgical intervention is the primary factor contributing to the unfavorable prognosis. Therefore, the critical aspect in improving the overall survival of HCC lies in reducing the postoperative recurrence rate. This review is aimed at summarizing the current evidence base regarding the safety and efficacy of systemic therapy administered in the neoadjuvant context for patients with resectable HCC. Furthermore, we will offer a perspective on the potential future trajectories of systemic therapy as a neoadjuvant modality in the management of HCC.', 'journal': 